# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [ ]:
#!pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [5]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [6]:
!pip install langchain_community


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [ ]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=1, model_name="gpt-6-luna")


In [40]:
prompt = ChatPromptTemplate.from_template( #Write a query that would take a variable to describe any product
  "Describe the following product: {product}"
)
prompt.invoke({"product": "iPhone 17"})

ChatPromptValue(messages=[HumanMessage(content='Describe the following product: iPhone 17')])

In [41]:

chain = prompt | llm

In [42]:
product = "Laptop"
response = chain.invoke({"product": product})
print(response)

content='A laptop is a portable personal computer with a built-in screen, keyboard, touchpad, and rechargeable battery. It lets you work, study, browse the web, stream media, and stay connected from home or on the go. Laptop models vary in performance, screen size, storage, and battery life.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 12, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 32, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-a4c40af1-b421-4f2f-ad76-40e0f344d286-0' usage_metadata={'input_tokens': 12, 'output_tokens': 102, 'total_tokens': 114}


## SimpleSequentialChain

In [43]:
from langchain.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    #Repeat the initial query or create a new query that would feed into the second prompt
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [ ]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    #Write the second prompt query that takes an input variable whose input will come from the previous prompt"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [ ]:
overall_simple_chain.run(product)

**Repeat the above twice for different products**

## SequentialChain

In [ ]:
from langchain.chains import SequentialChain

In [ ]:
llm = ChatOpenAI(temperature=1, model_name="gpt-6-luna")


first_prompt = ChatPromptTemplate.from_template(
  "Translate the following review into English:\n\n{review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="english_review"
                    )


In [59]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following review in one sentence:\n\n{english_review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [60]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "Translate the following summary into {language}:\n\n{summary}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="translated_summary"
                      )


In [61]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
         """Write a polite follow-up message to a customer
    based on the following information:

    Original review in English: {english_review}
    Review summary: {summary}

    Thank the customer for their feedback and
    address the main points in their review.
    """
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [62]:
from langchain.chains import SequentialChain
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review", "language"],
    output_variables=["english_review", "summary", "translated_summary", "followup_message"],
    verbose=False
)

In [63]:
review = df["Review"].iloc[5]

result = overall_chain.invoke({
    "review": review,
    "language": "German"
})

print(result)

{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?", 'language': 'German', 'english_review': 'I find the taste mediocre. The foam doesn’t last, which is strange. I buy the same ones in stores, and they taste much better...\n\nIs this an old batch or a counterfeit?!', 'summary': 'The product tastes worse than the same ones bought in stores, has foam that quickly disappears, and may be old or counterfeit.', 'translated_summary': 'Das Produkt schmeckt schlechter als dieselben Produkte aus dem Laden, der Schaum verschwindet schnell, und es könnte alt oder gefälscht sein.', 'followup_message': 'Thank you for sharing your feedback. We’re sorry the taste and foam didn’t meet your expectations, especially compared with the same product you’ve bought in stores. We understand why this would raise concerns about freshness or authenticity, and we’d like to look into it. Pl

**Repeat the above twice for different products or reviews**

In [79]:
for index in [2, 8]:
    review = df["Review"].iloc[index]

    result = overall_chain.invoke({
        "review": review,
        "language": "German"
    })

    print(f"\n--- Review {index} ---")
    print("Summary:", result["summary"])
    print("Follow-up:", result["followup_message"])


--- Review 2 ---
Summary: The mattress arrived with a hard-to-find hole that the patches couldn’t fix, leaving it flat and ruining a guest’s stay, while the required return shipment made the defective product even more frustrating.
Follow-up: Thank you for sharing your experience. We’re very sorry the mattress arrived with a hard-to-find hole, that the patches didn’t adhere to the fabric-like surface, and that it went flat during your friend’s stay. We understand how disappointing that was—and how frustrating it was to have to pack and ship the mattress back. This isn’t the quality or experience we want customers to have, and we’ll share your feedback with our quality and product teams. Please contact our customer service team with your order details so we can review what happened and see how we can help.


IndexError: single positional indexer is out-of-bounds

## Router Chain

In [64]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [65]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [66]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(temperature=1, model_name="gpt-6-luna")

In [69]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [70]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [71]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [72]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [73]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\rayha\AppData\Local\Temp\ipykernel_20456\3038952769.py:1: LangChainDeprecationWarning: Use RunnableLambda to select from multiple prompt templates. See example in API reference: https://api.python.langchain.com/en/latest/chains/langchain.chains.router.multi_prompt.MultiPromptChain.html
  chain = MultiPromptChain(router_chain=router_chain,


In [74]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black-body radiation is the electromagnetic radiation emitted by an ideal object that absorbs all radiation falling on it and emits radiation according to its temperature.\n\nIts spectrum depends only on the object’s temperature, not on what it’s made of. Cooler objects emit mostly infrared; hotter ones emit more visible light and can glow from red to white. A heated metal rod is a familiar approximation to a black body.'

In [75]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'2 + 2 = 4.'

In [76]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Most cells contain DNA because they inherited it from the fertilized egg. DNA provides the instructions a cell uses to make proteins and regulate its activities. As cells specialize—for example, becoming muscle or nerve cells—they usually keep the same basic genome but switch different genes on or off.\n\nBut **not every cell in the body contains DNA**. Mature red blood cells lose their nuclei as they develop, and platelets are cell fragments without nuclei. Most other cells retain DNA in a nucleus; many also have a small amount of DNA in their mitochondria.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [77]:
# Physics question
chain.invoke({
    "input": "Why is the sky blue?"
})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'Why is the sky blue?'}
> Finished chain.


{'input': 'Why is the sky blue?',
 'text': 'Sunlight contains all visible colors. As it passes through the atmosphere, tiny air molecules scatter the shorter wavelengths more strongly than the longer ones—a process called **Rayleigh scattering**. Blue light is scattered in all directions, so it reaches your eyes from across the sky.\n\nViolet light is scattered even more, but there’s less of it in sunlight and our eyes are less sensitive to it, so the sky looks blue rather than violet.'}

In [78]:
# History question
chain.invoke({
    "input": "What caused World War II?"
})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


History: {'input': 'What caused World War II?'}
> Finished chain.


{'input': 'What caused World War II?',
 'text': 'World War II did not have a single cause. It grew out of the collapse of the post–World War I international order, the rise of expansionist dictatorships, and the failure of other powers to stop aggression. The immediate trigger in Europe was Nazi Germany’s invasion of Poland on 1 September 1939.\n\n**The unsettled peace after World War I** created grievances and instability. The Treaty of Versailles imposed territorial losses, military restrictions and reparations on Germany. Many Germans regarded it as humiliating, and politicians exploited that resentment. But Versailles alone did not make another war inevitable: German governments had opportunities to pursue peaceful revision, and the treaty’s terms were modified during the 1920s. More broadly, the peace settlement left disputes unresolved and created new states with competing claims.\n\n**The rise of aggressive regimes** turned those tensions into a major danger. Adolf Hitler’s aims